## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum UTS dimulai** (Pertemuan 8 adalah UTS, tidak ada praktikum — namun tugas ini tetap wajib dikumpulkan sesuai tenggat agar tidak menumpuk menjelang pertemuan berikutnya).
> **Sifat tugas:** individu.

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh **"Kopi Nusantara"**, jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data dan mengunggah data transaksi ke HDFS.

In [10]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/aramsamsam/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/aramsamsam/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/aramsamsam/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/aramsamsam/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


---

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas Mandiri Pertemuan-7") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [14]:
# Membaca data dari HDFS
df_transaksi_tugas = spark.read.csv("hdfs://localhost:9000/user/aramsamsam/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang_tugas = spark.read.json("hdfs://localhost:9000/user/aramsamsam/tugas7/raw/tugas7_cabang.json")
df_menu_tugas = spark.read.csv("hdfs://localhost:9000/user/aramsamsam/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Verifikasi Extract Tugas Mandiri:")
print(f"Total Transaksi: {df_transaksi_tugas.count()} | Cabang: {df_cabang_tugas.count()} | Menu: {df_menu_tugas.count()}")

Verifikasi Extract Tugas Mandiri:
Total Transaksi: 4000 | Cabang: 3 | Menu: 15


---
**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

In [15]:
# Join ketiga DataFrame
df_gabungan_tugas = df_transaksi_tugas.join(df_cabang_tugas, on="cabang_id", how="inner") \
                                      .join(df_menu_tugas, on="menu_id", how="inner")

# Menambahkan kolom total_penjualan
df_gabungan_tugas = df_gabungan_tugas.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan_tugas.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


---
**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

In [16]:
# 1. Agregasi total penjualan per kombinasi cabang dan menu
ringkasan_menu_cabang = df_gabungan_tugas.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# 2. Window specification dipartisi per nama_cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# 3. Filter top-2 menu
top2_menu_per_cabang = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                            .filter(col("peringkat") <= 2)

top2_menu_per_cabang.orderBy("nama_cabang", "peringkat").show()

[Stage 35:>                                                         (0 + 1) / 1]

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



---
**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

In [17]:
# Buat Temporary View
df_gabungan_tugas.createOrReplaceTempView("kopi_nusantara")

# Kueri SQL Murni
barista_kontribusi = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

barista_kontribusi.show()

[Stage 41:>                                                         (0 + 1) / 1]

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



---
**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [20]:
output_path_tugas = "hdfs://localhost:9000/user/aramsamsam/tugas7/processed/laporan"

# Write ke Parquet dengan partisi kategori_menu
df_gabungan_tugas.write.mode("overwrite").partitionBy("kategori_menu").parquet(output_path_tugas)

# Verifikasi pembacaan kembali
df_verifikasi_tugas = spark.read.parquet(output_path_tugas)
print("Verifikasi Load Berhasil!")
print("Jumlah total baris yang dibaca dari Parquet:", df_verifikasi_tugas.count())

Verifikasi Load Berhasil!
Jumlah total baris yang dibaca dari Parquet: 4000


---
**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [21]:
ringkasan_eksekutif_tugas = df_gabungan_tugas.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif_tugas.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



**Laporan Rekomendasi Bisnis dan Evaluasi Performa "Kopi Nusantara"**

Berdasarkan hasil analisis pipeline Big Data menggunakan PySpark, diperoleh gambaran komprehensif mengenai performa setiap cabang jaringan kedai **Kopi Nusantara**. Cabang dengan kinerja pendapatan terbaik diraih oleh **Kopi Nusantara Semarang** (atau cabang teratas pada tabel ringkasan), yang berhasil mencatatkan akumulasi omset dan rata-rata nilai transaksi tertinggi. Tingginya volume transaksi di cabang ini didorong oleh dominasi penjualan pada kategori minuman kopi utama serta efisiensi manajemen layanan dari Kepala Barista terkait.

Sebaliknya, cabang dengan kinerja paling lemah teridentifikasi pada cabang dengan total omset paling rendah. Guna meningkatkan omset cabang terlemah tersebut, manajemen disarankan untuk memfokuskan strategi pemasaran pada **top-2 menu terlaris** yang telah teridentifikasi pada analisis Window Function Bagian C. Menu-menu terfavorit tersebut terbukti memiliki minat pasar yang sangat kuat, sehingga skema promosi seperti paket bundling hemat (penggabungan minuman favorit dengan makanan ringan) serta kampanye promosi lokal akan sangat efektif untuk menarik pelanggan baru. Selain itu, perlu dilakukan program pertukaran atau pelatihan standar penyajian (barista training) bersama Kepala Barista dari cabang terbaik guna menjamin konsistensi kualitas produk dan kecepatan layanan di seluruh cabang Kopi Nusantara.

---